# Demo - Anthropic SDK: Manually managed agent loop

## Setup

Install the Anthropic SDK package and dotenv so we can import an environment variable with our Anthropic API key

In [ ]:
!uv pip install anthropic dotenv


Import the `anthropic` package, create an API client, and define some constants (model, max iterations for this purpose of this demo).

In [ ]:
import anthropic
from dotenv import load_dotenv, find_dotenv

MODEL = "claude-sonnet-4-6"
MAX_ITERS = 6

# Import your Anthropic API key - ensure you have ANTHROPIC_API_KEY define in an .env file
load_dotenv(find_dotenv())

client = anthropic.Anthropic()

## Define tools

Define some mock tools which we can be used to simulate tool calls in the agent loop for the demo.

In [ ]:
# Tool schemas
TOOLS = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "input_schema": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    },
    {
        "name": "get_time",
        "description": "Get the current local time in a city.",
        "input_schema": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    },
]

# Mock tool implementations — static strings returned all the time
IMPL = {
    "get_weather": lambda city: f"12 degrees and clear in {city}",
    "get_time":    lambda city: f"7:42 pm in {city}",
}

## Prompt

This is the prompt which will be sent into the model.

We are calling the variable `msgs` as we will append the message/conversational history as the agent loop runs (in other words, holding all the previous context).

In [ ]:
msgs = [{
    "role": "user",
    "content": "What's the weather and the local time in Canberra?",
}]

## Helper function

This is a helper function to enable the tools to be called and a structured response be returned.

The response object from the Claude Messages API, which we pass into this function, has this structure:

```
Message(
	id='msg_011CdNggtw3DLnCZgZhSFFBS',
	container=None,
	content=[
		TextBlock(
			citations=None,
			text="I'll fetch the weather and local time for Canberra simultaneously!",
			type='text'
		),
		ToolUseBlock(
			id='toolu_018hGDyP5mErBvGrmu4NwCoM',
			caller=DirectCaller(type='direct'),
			input={'city': 'Canberra'},
			name='get_weather',
			type='tool_use'
		),
		ToolUseBlock(
			id='toolu_01ADE96pKuMf96jmdt2yfWAj',
			caller=DirectCaller(type='direct'),
			input={'city': 'Canberra'},
			name='get_time',
			type='tool_use'
		)
	],
	model='claude-sonnet-4-6',
	role='assistant',
	stop_details=None,
	stop_reason='tool_use',
	stop_sequence=None,
	type='message',
	usage=Usage(
		cache_creation=CacheCreation(
			ephemeral_1h_input_tokens=0,
			ephemeral_5m_input_tokens=0
		),
		cache_creation_input_tokens=0,
		cache_read_input_tokens=0,
		inference_geo='global',
		input_tokens=629,
		output_tokens=111,
		output_tokens_details=None,
		server_tool_use=None,
		service_tier='standard'
	)
)
```

In [ ]:
def run_tools(response):
    results = []
    for block in response.content:
        if block.type == "tool_use":
            print(f"    -> running {block.name}({block.input})")
            output = IMPL[block.name](**block.input)
            results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": output,
            })
    return results

## Define the agent loop

In [ ]:
# In reality, this would be an infinite loop, broken when the stop_reason == "end_turn"
# We are bounding it for the purpose of this demo so that when we do a demo of not checking for the stop_reason we don't have an infinite loop!
for _ in range(MAX_ITERS):

    # Create a request to the Messages API
    r = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        tools=TOOLS,
        messages=msgs,
    )

    # For demonstration, print what the current stop_reason value is
    print(f"stop_reason: {r.stop_reason}")

    # THIS IS WHAT YOU NEED TO CHECK FOR IN ORDER TO END THE AGENT LOOP
    # Note, you could also check for `max_tokens` and `stop_sequence`
    if r.stop_reason == "end_turn":
        text = "".join(b.text for b in r.content if b.type == "text")
        print(f'\nFinal response: "{text}"')
        break

    # If we reach here, we want to continue the agent loop (probably because stop_reason == tool_use
    # so let's append the model's response onto the conversational history (msg), run the tool, and append the
    # tool's response onto msg
    msgs.append({"role": "assistant", "content": r.content})
    msgs.append({"role": "user", "content": run_tools(r)})
else:
    print("\n...and it would do this forever. The loop lost its memory.")